In [1]:
import os

In [2]:
%pwd

'f:\\SCIENTIST\\DATA SCIENTIST PROJECTS\\AI-based-Kidney-Cancer-Detection-using-Deep-Learning-with-MLflow-and-DVC\\research'

In [3]:
os.chdir("../")

In [45]:
%pwd

'f:\\SCIENTIST\\DATA SCIENTIST PROJECTS\\AI-based-Kidney-Cancer-Detection-using-Deep-Learning-with-MLflow-and-DVC'

In [ ]:
# import dagshub
# dagshub.init(repo_owner='livin1991', repo_name='AI-based-Kidney-Cancer-Detection-using-Deep-Learning-with-MLflow-and-DVC', mlflow=True)

# import mlflow
# with mlflow.start_run():
#   mlflow.log_param('parameter name', 'value')
#   mlflow.log_metric('metric name', 1)

In [46]:
os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/livin1991/AI-based-Kidney-Cancer-Detection-using-Deep-Learning-with-MLflow-and-DVC.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="livin1991"
os.environ["MLFLOW_TRACKING_PASSWORD"]="5c5f09bf132f08d018d7ee7ab00e01bcf4c41af8"

In [ ]:
#set MLFLOW_TRACKING_URI=https://dagshub.com/livin1991/AI-based-Kidney-Cancer-Detection-using-Deep-Learning-with-MLflow-and-DVC.mlflow
# set MLFLOW_TRACKING_USERNAME=livin1991
# set MLFLOW_TRACKING_PASSWORD=5c5f09bf132f08d018d7ee7ab00e01bcf4c41af8

In [ ]:
# set MLFLOW_TRACKING_URI=https://dagshub.com/livin1991/AI-based-Kidney-Cancer-Detection-using-Deep-Learning-with-MLflow-and-DVC.mlflow
# set MLFLOW_TRACKING_USERNAME=livin1991 
# set MLFLOW_TRACKING_PASSWORD=5c5f09bf132f08d018d7ee7ab00e01bcf4c41af8

In [47]:
import tensorflow as tf

In [48]:
model = tf.keras.models.load_model("artifacts/training/model.h5")

In [49]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [50]:
from src.cnnClassifier.constants import *
from src.cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [51]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    
    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model="artifacts/training/model.h5",
            training_data="artifacts/data_ingestion/kidney-ct-scan-image",
            mlflow_uri="https://dagshub.com/livin1991/AI-based-Kidney-Cancer-Detection-using-Deep-Learning-with-MLflow-and-DVC.mlflow",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config

In [52]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse

In [ ]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    
    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )


    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = model.evaluate(self.valid_generator)
        self.save_score()

    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)

    
    def log_into_mlflow(self):
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics(
                {"loss": self.score[0], "accuracy": self.score[1]}
            )
            # Model registry does not work with file store
            if tracking_url_type_store != "file":

                # Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # please refer to the doc for more information:
                # https://mlflow.org/docs/latest/model-registry.html#api-workflow
                mlflow.keras.log_model(self.model, "model", registered_model_name="VGG16Model")
            else:
                mlflow.keras.log_model(self.model, "model")

In [57]:
import os
print(os.environ["MLFLOW_TRACKING_USERNAME"])
print(os.environ["MLFLOW_TRACKING_PASSWORD"])

livin1991
5c5f09bf132f08d018d7ee7ab00e01bcf4c41af8


In [56]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()

except Exception as e:
   raise e

[2026-03-19 22:36:38,719: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-19 22:36:38,728: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-19 22:36:38,732: INFO: common: created directory at: artifacts]
Found 139 images belonging to 2 classes.
9/9 [==============================] - 75s 8s/step - loss: 1.9197 - accuracy: 0.5468
[2026-03-19 22:37:55,065: INFO: common: json file saved at: scores.json]


RestException: INVALID_PARAMETER_VALUE: Response: {'error_code': 'INVALID_PARAMETER_VALUE'}